### 04. 직렬화와 역질렬화로 모델 저장 및 로드하기 : page 205

In [1]:
# !pip --version

pip 25.0.1 from D:\skc0902\ex0917\.venv\Lib\site-packages\pip (python 3.12)



In [1]:
# !pip install dotenv
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

True

In [28]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.load import dumpd, dumps
import pickle
import json
from langchain_core.load import load, loads

In [6]:
prompt = PromptTemplate.from_template("{fruit}의 색상이 무엇입니까?")
prompt

PromptTemplate(input_variables=['fruit'], input_types={}, partial_variables={}, template='{fruit}의 색상이 무엇입니까?')

In [ ]:
# 직렬화 체크
print(f"ChatOpenAI: {ChatOpenAI.is_lc_serializable()}")

ChatOpenAI: True


In [8]:
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# llm 직렬화 체크.
print(f"ChatOpenAI: {llm.is_lc_serializable()}")

ChatOpenAI: True


In [9]:
chain = prompt | llm

# chain 직렬화 가능 체크
chain.is_lc_serializable()

True

In [ ]:
# dumpd: chain오브젝트를 dict로 직렬화
dumpd_chain = dumpd(chain)
dumpd_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-4.1-mini',
    'temperature': 0.0,
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [12]:
type(dumpd_chain)

dict

In [21]:
# dumps: 객체를 문자열로 직렬화
dumps_chain = dumps(chain)
dumps_chain

'{"lc": 1, "type": "constructor", "id": ["langchain", "schema", "runnable", "RunnableSequence"], "kwargs": {"first": {"lc": 1, "type": "constructor", "id": ["langchain", "prompts", "prompt", "PromptTemplate"], "kwargs": {"input_variables": ["fruit"], "template": "{fruit}\\uc758 \\uc0c9\\uc0c1\\uc774 \\ubb34\\uc5c7\\uc785\\ub2c8\\uae4c?", "template_format": "f-string"}, "name": "PromptTemplate"}, "last": {"lc": 1, "type": "constructor", "id": ["langchain", "chat_models", "openai", "ChatOpenAI"], "kwargs": {"model_name": "gpt-4.1-mini", "temperature": 0.0, "openai_api_key": {"lc": 1, "type": "secret", "id": ["OPENAI_API_KEY"]}, "stream_usage": true}, "name": "ChatOpenAI"}}, "name": "RunnableSequence"}'

In [22]:
type(dumps_chain)

str

### Pickle 파일
-  Python 객체를 바이너리 형태로 직렬화하는 포맷

In [ ]:
# 파일이 없으면 생성후, 바이너리형식으로 쓰기 수행.
with open("fruit_chain.pkl", "wb") as f:
    # pickle.dump(): chain오브젝트를 파일에 저장
    pickle.dump(dumpd_chain, f)

In [ ]:
# 파일이 없으면 생성후, json형식으로 쓰기 수행.
with open("fruit_chain.json", "w") as fp:
    # json.dump(): chain오브젝트를 파일에 저장
    json.dump(dumpd_chain, fp)

In [23]:
# 저장한 pickle 형식의 파일을 가져오기
with open("fruit_chain.pkl", "rb") as f:
    loaded_chain = pickle.load(f)

In [24]:
loaded_chain

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'],
 'kwargs': {'first': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'],
   'kwargs': {'input_variables': ['fruit'],
    'template': '{fruit}의 색상이 무엇입니까?',
    'template_format': 'f-string'},
   'name': 'PromptTemplate'},
  'last': {'lc': 1,
   'type': 'constructor',
   'id': ['langchain', 'chat_models', 'openai', 'ChatOpenAI'],
   'kwargs': {'model_name': 'gpt-4.1-mini',
    'temperature': 0.0,
    'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['OPENAI_API_KEY']},
    'stream_usage': True},
   'name': 'ChatOpenAI'}},
 'name': 'RunnableSequence'}

In [ ]:
# 역직열화로 복원된 파일 : 단순 체인으로 복원됨.

# 체인 로드
chain_from_file = load(loaded_chain, allowed_objects="all")

# 체인 실행.
print(chain_from_file.invoke({"fruit": "바나나"}))

content='바나나의 색상은 일반적으로 노란색입니다. 익지 않은 바나나는 초록색을 띠고, 완전히 익으면 노란색이 되며, 너무 익으면 갈색 반점이 생기기도 합니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 17, 'total_tokens': 69, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_e941d48efd', 'id': 'chatcmpl-EOzKNrW7REVWn75AYC0DoxwljZXWl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0adec-0811-7d50-a0db-4f27dbe23a96-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 17, 'output_tokens': 52, 'total_tokens': 69, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'

In [ ]:
# 역직열화로 복원된 파일 :  단순 체인으로 복원후, API Key 같은 Secret이 포함된 체인 로드.

load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["OPENAI_API_KEY"]},
    allowed_objects="all"
)

# 체인 정상동작 확인
load_chain.invoke({"fruit": "오렌지"})

AIMessage(content='오렌지의 색상은 주로 주황색입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 17, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_7681b9e269', 'id': 'chatcmpl-EOzFkOoW3KX5yTAjgdfmbB9wDdTfw', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ade7-a935-77c3-8016-96677064be69-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 17, 'output_tokens': 14, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [36]:
with open("fruit_chain.json", "r") as fp:
    loaded_from_json_chain = json.load(fp)
    loads_chain = load(loaded_from_json_chain, allowed_objects="all")

# 불러온 체인 정상 동작 확인
loads_chain.invoke({"fruit": "사과"})

AIMessage(content='사과의 색상은 보통 빨간색, 초록색, 노란색 등이 있습니다. 품종에 따라 다양한 색상이 나타날 수 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 16, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_117d60f06a', 'id': 'chatcmpl-EP0Gr6S8R1cJGqYTIc1r8XSsyiunL', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ae23-5b3d-73f0-b532-d5de8ba4c375-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 16, 'output_tokens': 34, 'total_tokens': 50, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio':

In [ ]:
# pickle.dump(): 객체를 파일에 저장